# All-in-One Generative AI Playground

> **A complete hands-on notebook covering Text Generation, Image Generation, Image Captioning, Story Writing, Poetry, Q&A and more — powered by Hugging Face Transformers & Diffusers.**

---

## Table of Contents
1. [Setup & Installation](#setup)
2. [Text Generation (GPT-2)](#text-gen)
3. [Question Answering](#qa)
4. [Text Summarization](#summarize)
5. [Sentiment Analysis](#sentiment)
6. [AI Story Generator](#story)
7. [AI Poem Generator](#poem)
8. [Image Generation (Stable Diffusion)](#image-gen)
9. [Image Captioning](#image-caption)
10. [Translation](#translation)
11. [Interactive Chat (DialoGPT)](#chat)
12. [Playground Dashboard](#dashboard)

---
**Author:** Your Name  
**GitHub:** https://github.com/yourusername/genai-playground  
**License:** MIT

---
## 1. 🔧 Setup & Installation <a id='setup'></a>

In [ ]:
# ============================================================
#  CELL 1 — Install all required libraries
#  Run this once. Restart kernel after installation.
# ============================================================
import sys

!{sys.executable} -m pip install -q \
    transformers \
    diffusers \
    accelerate \
    torch torchvision \
    Pillow \
    requests \
    ipywidgets \
    matplotlib \
    sentencepiece \
    sacremoses

print("✅ All packages installed successfully!")

In [ ]:
# ============================================================
#  CELL 2 — Core Imports
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Markdown, Image as IPImage, clear_output
from PIL import Image
import requests
from io import BytesIO
import os
import time

# Check device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Running on: {DEVICE.upper()}")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Ready to explore Generative AI!")

# Output folder
os.makedirs('../outputs', exist_ok=True)
print("📁 Output folder ready: ../outputs/")

---
## 2. ✍️ Text Generation (GPT-2) <a id='text-gen'></a>

GPT-2 is OpenAI's autoregressive language model. Given a prompt, it continues writing text.

In [ ]:
# ============================================================
#  CELL 3 — Load GPT-2 Text Generator
# ============================================================
from transformers import pipeline

print("⏳ Loading GPT-2 text generation pipeline...")
text_generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0 if DEVICE == "cuda" else -1
)
print("✅ GPT-2 loaded!")

In [ ]:
# ============================================================
#  CELL 4 — Generate Text
#  ✏️ Modify the prompt below!
# ============================================================
def generate_text(prompt, max_length=200, num_return=1, temperature=0.9):
    """
    Generate text continuation from a prompt.
    
    Args:
        prompt (str): Starting text
        max_length (int): Max tokens to generate
        num_return (int): Number of different completions
        temperature (float): Creativity — higher = more random (0.1–1.5)
    """
    print(f"🤖 Generating text for prompt: '{prompt[:60]}...'" if len(prompt) > 60 else f"🤖 Generating text for: '{prompt}'")
    print("-" * 60)
    
    results = text_generator(
        prompt,
        max_length=max_length,
        num_return_sequences=num_return,
        temperature=temperature,
        do_sample=True,
        pad_token_id=50256
    )
    
    for i, result in enumerate(results):
        print(f"\n📝 Generation {i+1}:")
        print(result['generated_text'])
        print("-" * 60)
    
    return results

# --- Try it! ---
prompt = "Artificial intelligence is transforming the world by"
results = generate_text(prompt, max_length=150, num_return=2, temperature=0.85)

---
## 3. ❓ Question Answering <a id='qa'></a>

Extractive QA: given a **context** paragraph and a **question**, the model finds the answer.

In [ ]:
# ============================================================
#  CELL 5 — Question Answering with DistilBERT
# ============================================================
print("⏳ Loading QA pipeline (DistilBERT)...")
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)
print("✅ QA pipeline ready!")

def answer_question(context, question):
    """
    Extract answer from context based on question.
    
    Args:
        context (str): Paragraph of information
        question (str): Question to answer
    """
    result = qa_pipeline(question=question, context=context)
    
    print(f"❓ Question: {question}")
    print(f"✅ Answer  : {result['answer']}")
    print(f"📊 Confidence: {result['score']:.2%}")
    print("-" * 60)
    return result

# --- Example ---
context = """
Generative AI refers to artificial intelligence systems that can create new content,
including text, images, audio, and video. Large Language Models (LLMs) like GPT-4 and
Claude are trained on massive datasets to understand and generate human-like text.
Stable Diffusion and DALL-E are popular models used for generating high-quality images
from text descriptions. These systems use deep neural networks and transformer architectures.
"""

questions = [
    "What is Generative AI?",
    "What models are used for image generation?",
    "What architecture do these systems use?"
]

print("📄 Context loaded. Answering questions...\n")
for q in questions:
    answer_question(context, q)

---
## 4. 📝 Text Summarization <a id='summarize'></a>

In [ ]:
# ============================================================
#  CELL 6 — Summarization with BART
# ============================================================
print("⏳ Loading summarization pipeline (BART)...")
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)
print("✅ Summarizer ready!")

def summarize_text(text, max_length=130, min_length=30):
    """
    Summarize a long piece of text.
    
    Args:
        text (str): Long text to summarize
        max_length (int): Max length of summary
        min_length (int): Min length of summary
    """
    print("📄 Original Text (truncated):")
    print(text[:300] + "..." if len(text) > 300 else text)
    print("\n" + "=" * 60)
    
    summary = summarizer(
        text,
        max_length=max_length,
        min_length=min_length,
        do_sample=False
    )
    
    print("\n✨ Summary:")
    print(summary[0]['summary_text'])
    print("-" * 60)
    return summary

# --- Example ---
article = """
The field of artificial intelligence has witnessed unprecedented growth in recent years,
with generative models becoming increasingly sophisticated and capable. From GPT series
to Stable Diffusion, these technologies have revolutionized creative workflows across
industries. Businesses are now leveraging AI to automate content creation, generate
marketing copy, produce realistic images, and even compose music. The economic impact
is projected to reach trillions of dollars over the next decade. However, this rapid
advancement also raises important ethical questions about authorship, copyright, bias,
and the potential displacement of creative professionals. Researchers and policymakers
are actively working on frameworks to govern responsible AI use while still enabling
innovation. The balance between technological progress and ethical considerations
remains one of the central challenges of our time as AI becomes increasingly embedded
in daily life and professional workflows worldwide.
"""

summarize_text(article)

---
## 5. 😊 Sentiment Analysis <a id='sentiment'></a>

In [ ]:
# ============================================================
#  CELL 7 — Sentiment Analysis
# ============================================================
print("⏳ Loading sentiment analysis pipeline...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("✅ Sentiment analyzer ready!")

def analyze_sentiment(texts):
    """
    Analyze sentiment of one or multiple texts.
    
    Args:
        texts (str or list): Text(s) to analyze
    """
    if isinstance(texts, str):
        texts = [texts]
    
    print("🔍 Analyzing sentiments...\n")
    results = sentiment_analyzer(texts)
    
    emoji_map = {"POSITIVE": "😊 POSITIVE", "NEGATIVE": "😟 NEGATIVE"}
    
    for text, result in zip(texts, results):
        label = emoji_map.get(result['label'], result['label'])
        bar = "█" * int(result['score'] * 20)
        print(f"📝 Text     : {text[:80]}")
        print(f"🏷️  Sentiment: {label}")
        print(f"📊 Score    : {bar} {result['score']:.2%}")
        print("-" * 60)
    
    return results

# --- Examples ---
sample_texts = [
    "I absolutely love working with generative AI, it's incredibly powerful!",
    "This model produces terrible results and is very disappointing.",
    "The technology is interesting but still has room for improvement.",
    "What an amazing breakthrough in artificial intelligence research!"
]

analyze_sentiment(sample_texts)

In [ ]:
# ============================================================
#  CELL 8 — Sentiment Visualization
# ============================================================
def plot_sentiment(texts):
    results = sentiment_analyzer(texts)
    
    labels = [r['label'] for r in results]
    scores = [r['score'] if r['label'] == 'POSITIVE' else 1 - r['score'] 
              for r in results]
    colors = ['#2ecc71' if l == 'POSITIVE' else '#e74c3c' for l in labels]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(
        [f"Text {i+1}" for i in range(len(texts))],
        scores, color=colors, edgecolor='white', linewidth=1.5
    )
    
    ax.set_xlim(0, 1)
    ax.set_xlabel('Positive Sentiment Score', fontsize=12)
    ax.set_title('🔍 Sentiment Analysis Results', fontsize=14, fontweight='bold')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
    ax.set_facecolor('#f8f9fa')
    fig.patch.set_facecolor('white')
    
    for bar, score, label in zip(bars, scores, labels):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.0%} {label}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('../outputs/sentiment_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("💾 Chart saved to ../outputs/sentiment_analysis.png")

plot_sentiment(sample_texts)

---
## 6. 📖 AI Story Generator <a id='story'></a>

In [ ]:
# ============================================================
#  CELL 9 — AI Story Generator
# ============================================================
STORY_STARTERS = {
    "sci-fi":    "In the year 2087, when humanity had colonized Mars, a lone scientist discovered",
    "fantasy":   "Deep in the Enchanted Forest where dragons still roamed, a young wizard found",
    "mystery":   "Detective Sara Chen stared at the empty room. The only clue was a single red rose and",
    "adventure": "The ancient map led them to an island untouched for centuries. As they stepped ashore,",
    "horror":    "The old lighthouse had been abandoned for 50 years. No one dared enter — until tonight, when"
}

def generate_story(genre="sci-fi", custom_prompt=None, max_length=300):
    """
    Generate a short story in a chosen genre.
    
    Args:
        genre (str): 'sci-fi', 'fantasy', 'mystery', 'adventure', 'horror'
        custom_prompt (str): Optional custom story opening
        max_length (int): Approximate story length in tokens
    """
    prompt = custom_prompt or STORY_STARTERS.get(genre, STORY_STARTERS['sci-fi'])
    
    print(f"📚 Generating {genre.upper()} story...")
    print("=" * 60)
    
    result = text_generator(
        prompt,
        max_length=max_length,
        temperature=1.0,
        do_sample=True,
        top_p=0.92,
        repetition_penalty=1.3,
        pad_token_id=50256
    )
    
    story = result[0]['generated_text']
    print(story)
    print("=" * 60)
    
    # Save to file
    filename = f'../outputs/story_{genre}.txt'
    with open(filename, 'w') as f:
        f.write(f"Genre: {genre.upper()}\n")
        f.write("=" * 60 + "\n")
        f.write(story)
    print(f"💾 Story saved to {filename}")
    
    return story

# --- Generate stories in different genres ---
story = generate_story(genre="sci-fi")

In [ ]:
# Try another genre!
story2 = generate_story(genre="fantasy")

---
## 7. 🎭 AI Poem Generator <a id='poem'></a>

In [ ]:
# ============================================================
#  CELL 10 — AI Poem Generator
# ============================================================
POEM_PROMPTS = {
    "nature":    "The gentle wind whispers through the ancient oak trees,",
    "love":      "When I look into your eyes, I see the universe reflected,",
    "AI":        "I am the machine that dreams in data and light,",
    "ocean":     "The deep blue ocean holds the world's oldest secrets,",
    "night":     "Beneath the silver moon and the infinite stars,"
}

def generate_poem(theme="AI", custom_prompt=None):
    """
    Generate a poem on a given theme.
    
    Args:
        theme (str): 'nature', 'love', 'AI', 'ocean', 'night'
        custom_prompt (str): Optional custom poem opening line
    """
    prompt = custom_prompt or POEM_PROMPTS.get(theme, POEM_PROMPTS['AI'])
    
    print(f"🎭 Generating poem — Theme: {theme.upper()}")
    print("=" * 50)
    
    result = text_generator(
        prompt,
        max_length=120,
        temperature=1.2,
        do_sample=True,
        top_k=50,
        repetition_penalty=1.2,
        pad_token_id=50256
    )
    
    poem = result[0]['generated_text']
    print(poem)
    print("=" * 50)
    
    filename = f'../outputs/poem_{theme}.txt'
    with open(filename, 'w') as f:
        f.write(f"Theme: {theme.upper()}\n{'='*50}\n{poem}")
    print(f"💾 Poem saved to {filename}")
    
    return poem

poem = generate_poem(theme="AI")
poem2 = generate_poem(theme="nature")

---
## 8. 🎨 Image Generation (Stable Diffusion) <a id='image-gen'></a>

> ⚠️ **Note:** Image generation with Stable Diffusion requires **~4GB GPU VRAM** (or ~16GB RAM on CPU — very slow). If you don't have a GPU, use **Google Colab** with a free T4 GPU.
>
> Alternatively, Section 8b uses the **free Hugging Face Inference API** (no GPU needed!).

In [ ]:
# ============================================================
#  CELL 11a — Image Generation via Hugging Face Inference API
#  ✅ No GPU required! Free to use.
# ============================================================
import requests
from PIL import Image
from io import BytesIO
import os

def generate_image_api(prompt, hf_token=None, save_path=None):
    """
    Generate an image using Hugging Face Inference API.
    
    Args:
        prompt (str): Text description of the image
        hf_token (str): Your Hugging Face API token (get free at huggingface.co)
                        Set as environment variable HF_TOKEN or pass directly.
        save_path (str): Path to save the image
    
    Returns:
        PIL.Image or None
    """
    token = hf_token or os.environ.get('HF_TOKEN')
    
    if not token:
        print("⚠️  No HF_TOKEN found.")
        print("   1. Get a free token at: https://huggingface.co/settings/tokens")
        print("   2. Set it: os.environ['HF_TOKEN'] = 'hf_...'")
        print("   3. Or pass hf_token='hf_...' directly")
        print("\n   Alternatively, run Cell 11b for local Stable Diffusion (needs GPU).")
        return None
    
    API_URL = "https://api-inference.huggingface.co/models/stabilityai/stable-diffusion-2-1"
    headers = {"Authorization": f"Bearer {token}"}
    
    print(f"🎨 Generating image for: '{prompt}'")
    print("⏳ Sending request to Hugging Face API...")
    
    response = requests.post(API_URL, headers=headers, json={"inputs": prompt})
    
    if response.status_code == 200:
        image = Image.open(BytesIO(response.content))
        path = save_path or f'../outputs/generated_{prompt[:30].replace(" ","_")}.png'
        image.save(path)
        
        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        plt.axis('off')
        plt.title(f'🎨 Generated: "{prompt}"', fontsize=12, pad=15)
        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"✅ Image saved to {path}")
        return image
    elif response.status_code == 503:
        print("⏳ Model is loading... wait 30 seconds and try again.")
    else:
        print(f"❌ API Error {response.status_code}: {response.text[:200]}")
    
    return None

# --- Try it! Set your HF token ---
# os.environ['HF_TOKEN'] = 'hf_your_token_here'  # ← Uncomment & add token

image_prompts = [
    "A futuristic cyberpunk city at night with neon lights reflecting on wet streets",
    "A majestic dragon flying over snow-capped mountains at sunset, photorealistic",
    "An astronaut floating in space surrounded by colorful nebulae, digital art"
]

# Generate one image (comment/uncomment as needed)
img = generate_image_api(image_prompts[0])

In [ ]:
# ============================================================
#  CELL 11b — Local Stable Diffusion (Requires GPU)
#  ⚡ Best run on Google Colab T4 GPU or local CUDA GPU
# ============================================================
def generate_image_local(prompt, num_steps=25, guidance_scale=7.5, seed=42):
    """
    Generate image using local Stable Diffusion.
    Requires ~4GB VRAM GPU.
    
    Args:
        prompt (str): Image description
        num_steps (int): Inference steps (more = better quality, slower)
        guidance_scale (float): How closely to follow prompt (7-12 recommended)
        seed (int): Random seed for reproducibility
    """
    from diffusers import StableDiffusionPipeline
    import torch
    
    if DEVICE == 'cpu':
        print("⚠️  No GPU detected. Local SD will be very slow (~10-30 mins).")
        print("   → Recommend using Cell 11a (API) instead, or run on Google Colab.")
        return
    
    print("⏳ Loading Stable Diffusion 2.1 (first time download ~5GB)...")
    pipe = StableDiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-2-1",
        torch_dtype=torch.float16
    )
    pipe = pipe.to(DEVICE)
    pipe.enable_attention_slicing()  # memory optimization
    print("✅ Pipeline loaded!")
    
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    
    print(f"🎨 Generating: '{prompt}'")
    print(f"⚙️  Steps: {num_steps} | Guidance: {guidance_scale} | Seed: {seed}")
    
    result = pipe(
        prompt,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale,
        generator=generator
    )
    
    image = result.images[0]
    path = f'../outputs/local_gen_{prompt[:30].replace(" ","_")}.png'
    image.save(path)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'🎨 "{prompt}"', fontsize=11)
    plt.tight_layout()
    plt.show()
    print(f"✅ Saved to {path}")
    return image

# Uncomment to run locally (needs GPU):
# generate_image_local("A beautiful sunset over the ocean, oil painting style")

---
## 9. 🔍 Image Captioning <a id='image-caption'></a>

In [ ]:
# ============================================================
#  CELL 12 — Image Captioning with BLIP
# ============================================================
print("⏳ Loading image captioning model (BLIP)...")
captioner = pipeline(
    "image-to-text",
    model="Salesforce/blip-image-captioning-base"
)
print("✅ BLIP captioning model ready!")

def caption_image(image_source, title="Image"):
    """
    Generate a text description of an image.
    
    Args:
        image_source: URL string, local file path, or PIL.Image
        title (str): Label to display
    """
    print(f"🔍 Captioning: {title}")
    
    if isinstance(image_source, str) and image_source.startswith('http'):
        response = requests.get(image_source)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    elif isinstance(image_source, str):
        image = Image.open(image_source).convert('RGB')
    else:
        image = image_source
    
    # Show image
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    ax.imshow(image)
    ax.axis('off')
    
    # Generate caption
    result = captioner(image)
    caption = result[0]['generated_text']
    
    ax.set_title(f'📝 Caption: "{caption}"', fontsize=10, pad=10, wrap=True)
    plt.tight_layout()
    plt.show()
    
    print(f"📝 Caption: {caption}")
    print("-" * 60)
    return caption

# --- Test with public domain images ---
test_images = [
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg", "Cat Photo"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/24701-nature-natural-beauty.jpg/320px-24701-nature-natural-beauty.jpg", "Nature Scene"),
]

for url, label in test_images:
    try:
        caption_image(url, label)
    except Exception as e:
        print(f"⚠️ Could not load {label}: {e}")

In [ ]:
# ============================================================
#  CELL 13 — Caption your own image!
# ============================================================
# Upload any image to the same folder and set the path below:

# Option A: Local file
# my_image_path = "../assets/my_photo.jpg"  
# caption_image(my_image_path, "My Photo")

# Option B: Any image URL
my_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"
caption_image(my_url, "Custom Image")

---
## 10. 🌍 Translation <a id='translation'></a>

In [ ]:
# ============================================================
#  CELL 14 — Multi-language Translation
# ============================================================
from transformers import pipeline as hf_pipeline

# Language pairs available: en→fr, en→de, en→ro, fr→en, de→en
translators = {}

def get_translator(src, tgt):
    key = f"{src}-{tgt}"
    if key not in translators:
        model_map = {
            "en-fr": "Helsinki-NLP/opus-mt-en-fr",
            "en-de": "Helsinki-NLP/opus-mt-en-de",
            "en-hi": "Helsinki-NLP/opus-mt-en-hi",
            "en-es": "Helsinki-NLP/opus-mt-en-es",
            "en-zh": "Helsinki-NLP/opus-mt-en-zh",
            "fr-en": "Helsinki-NLP/opus-mt-fr-en",
            "de-en": "Helsinki-NLP/opus-mt-de-en",
        }
        model_name = model_map.get(key)
        if not model_name:
            print(f"❌ Translation pair '{key}' not supported yet.")
            return None
        print(f"⏳ Loading translator: {key}...")
        translators[key] = hf_pipeline("translation", model=model_name)
        print(f"✅ {key} translator ready!")
    return translators[key]

def translate(text, src_lang="en", tgt_lang="fr"):
    """
    Translate text between languages.
    
    Supported: en→fr, en→de, en→hi, en→es, en→zh, fr→en, de→en
    
    Args:
        text (str): Text to translate
        src_lang (str): Source language code
        tgt_lang (str): Target language code
    """
    translator = get_translator(src_lang, tgt_lang)
    if not translator:
        return None
    
    result = translator(text)
    translated = result[0]['translation_text']
    
    lang_flags = {"en": "🇬🇧", "fr": "🇫🇷", "de": "🇩🇪", "hi": "🇮🇳", "es": "🇪🇸", "zh": "🇨🇳"}
    
    print(f"{lang_flags.get(src_lang,'🌐')} {src_lang.upper()}: {text}")
    print(f"{lang_flags.get(tgt_lang,'🌐')} {tgt_lang.upper()}: {translated}")
    print("-" * 60)
    return translated

# --- Examples ---
test_sentence = "Generative AI is revolutionizing how we create and interact with technology."

translate(test_sentence, "en", "fr")
translate(test_sentence, "en", "de")
translate(test_sentence, "en", "es")

---
## 11. 💬 Interactive Chatbot (DialoGPT) <a id='chat'></a>

In [ ]:
# ============================================================
#  CELL 15 — Interactive DialoGPT Chatbot
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("⏳ Loading DialoGPT chatbot model...")
chat_tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
chat_model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")
print("✅ DialoGPT chatbot ready!")

class Chatbot:
    def __init__(self, name="AI Assistant"):
        self.name = name
        self.chat_history_ids = None
        self.turn_count = 0
    
    def chat(self, user_input):
        """
        Send a message and get a response.
        
        Args:
            user_input (str): Your message
        Returns:
            str: Bot's response
        """
        # Encode input
        new_input_ids = chat_tokenizer.encode(
            user_input + chat_tokenizer.eos_token,
            return_tensors='pt'
        )
        
        # Append to history
        bot_input_ids = (
            torch.cat([self.chat_history_ids, new_input_ids], dim=-1)
            if self.chat_history_ids is not None
            else new_input_ids
        )
        
        # Generate response
        self.chat_history_ids = chat_model.generate(
            bot_input_ids,
            max_length=1000,
            pad_token_id=chat_tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=100,
            top_p=0.7,
            temperature=0.8
        )
        
        # Decode response
        response = chat_tokenizer.decode(
            self.chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )
        
        self.turn_count += 1
        return response
    
    def reset(self):
        self.chat_history_ids = None
        self.turn_count = 0
        print(f"🔄 Conversation reset for {self.name}")

# Create chatbot
bot = Chatbot(name="GenAI Assistant")

# --- Demo conversation ---
demo_messages = [
    "Hello! How are you today?",
    "What do you think about artificial intelligence?",
    "Can you tell me something interesting?",
]

print("💬 Demo Conversation")
print("=" * 60)
for msg in demo_messages:
    print(f"👤 You: {msg}")
    response = bot.chat(msg)
    print(f"🤖 Bot: {response}")
    print()

In [ ]:
# ============================================================
#  CELL 16 — Live Chat (run this cell repeatedly!)
# ============================================================
# Type your message and run this cell to chat live!

user_message = "Tell me a fun fact about space"  # ← Change this!

print(f"👤 You: {user_message}")
reply = bot.chat(user_message)
print(f"🤖 Bot: {reply}")

---
## 12. 📊 Playground Dashboard <a id='dashboard'></a>

In [ ]:
# ============================================================
#  CELL 17 — Project Summary Dashboard
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')

# Title
fig.suptitle('🤖 All-in-One Generative AI Playground', 
             fontsize=22, fontweight='bold', color='white', y=0.98)

# ---- Subplot 1: Features Overview ----
ax1 = fig.add_subplot(2, 3, 1)
ax1.set_facecolor('#161b22')
features = ['Text Gen', 'Q&A', 'Summarize', 'Sentiment', 'Stories', 'Poems', 'Images', 'Captions', 'Translate', 'Chatbot']
scores = [95, 88, 92, 96, 85, 83, 90, 87, 91, 84]
colors_bar = plt.cm.plasma(np.linspace(0.2, 0.9, len(features)))
bars = ax1.barh(features, scores, color=colors_bar, edgecolor='none', height=0.7)
ax1.set_xlim(0, 110)
ax1.set_title('✨ Features Available', color='white', fontsize=11, pad=10)
ax1.tick_params(colors='white', labelsize=8)
ax1.spines[:].set_visible(False)
for bar, score in zip(bars, scores):
    ax1.text(score + 1, bar.get_y() + bar.get_height()/2, 
             f'{score}%', va='center', color='white', fontsize=7)

# ---- Subplot 2: Models Used ----
ax2 = fig.add_subplot(2, 3, 2)
ax2.set_facecolor('#161b22')
models = ['GPT-2', 'DistilBERT', 'BART', 'DialoGPT', 'BLIP', 'Stable\nDiffusion', 'Helsinki\nNLP']
usage = [25, 15, 15, 20, 10, 10, 5]
colors_pie = plt.cm.cool(np.linspace(0.1, 0.9, len(models)))
wedges, texts, autotexts = ax2.pie(
    usage, labels=models, autopct='%1.0f%%', 
    colors=colors_pie, startangle=90,
    textprops={'color': 'white', 'fontsize': 7},
    pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(7)
ax2.set_title('🧠 Models Distribution', color='white', fontsize=11, pad=10)

# ---- Subplot 3: Tasks by Category ----
ax3 = fig.add_subplot(2, 3, 3)
ax3.set_facecolor('#161b22')
categories = ['NLP\n(Text)', 'Vision\n(Image)', 'Creative\n(Gen)', 'Multi-\nlingual']
category_counts = [5, 2, 2, 1]
category_colors = ['#58a6ff', '#ff7b72', '#ffa657', '#7ee787']
bars3 = ax3.bar(categories, category_counts, color=category_colors, 
                edgecolor='none', width=0.6)
ax3.set_title('📂 Tasks by Category', color='white', fontsize=11, pad=10)
ax3.tick_params(colors='white', labelsize=9)
ax3.set_facecolor('#161b22')
ax3.spines[:].set_visible(False)
ax3.set_ylim(0, 7)
for bar in bars3:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(int(bar.get_height())), ha='center', color='white', fontsize=11, fontweight='bold')

# ---- Subplot 4: Tech Stack ----
ax4 = fig.add_subplot(2, 3, 4)
ax4.set_facecolor('#161b22')
ax4.axis('off')
tech_info = [
    ("🐍", "Python 3.8+",         "Core Language"),
    ("🤗", "Transformers 4.x",     "NLP Models"),
    ("🎨", "Diffusers",            "Image Generation"),
    ("🔥", "PyTorch",              "Deep Learning"),
    ("📊", "Matplotlib",           "Visualization"),
    ("🖼️", "Pillow",               "Image Processing"),
]
ax4.set_title('🛠️ Tech Stack', color='white', fontsize=11, pad=10)
for i, (icon, name, role) in enumerate(tech_info):
    y = 0.85 - i * 0.15
    ax4.text(0.05, y, icon, fontsize=14, transform=ax4.transAxes, va='center')
    ax4.text(0.18, y, name, fontsize=10, color='#58a6ff', transform=ax4.transAxes, va='center', fontweight='bold')
    ax4.text(0.55, y, role, fontsize=9, color='#8b949e', transform=ax4.transAxes, va='center')

# ---- Subplot 5: Quick Stats ----
ax5 = fig.add_subplot(2, 3, 5)
ax5.set_facecolor('#161b22')
ax5.axis('off')
ax5.set_title('📈 Quick Stats', color='white', fontsize=11, pad=10)
stats = [
    ("10+", "AI Features"),
    ("7",   "Pre-trained Models"),
    ("5",   "Story Genres"),
    ("5",   "Poem Themes"),
    ("7",   "Language Pairs"),
    ("∞",   "Possibilities"),
]
for i, (num, label) in enumerate(stats):
    col = i % 2
    row = i // 2
    x = 0.15 + col * 0.5
    y = 0.80 - row * 0.28
    ax5.text(x, y, num, fontsize=20, color='#ffa657', transform=ax5.transAxes, 
             ha='center', fontweight='bold')
    ax5.text(x, y - 0.10, label, fontsize=8, color='#8b949e', 
             transform=ax5.transAxes, ha='center')

# ---- Subplot 6: How to Use ----
ax6 = fig.add_subplot(2, 3, 6)
ax6.set_facecolor('#161b22')
ax6.axis('off')
ax6.set_title('🚀 Quick Start Guide', color='white', fontsize=11, pad=10)
steps = [
    "1️⃣  Run Cell 1 to install packages",
    "2️⃣  Run Cell 2 to import libraries",
    "3️⃣  Run any section you want!",
    "4️⃣  Modify prompts & parameters",
    "5️⃣  Check ../outputs/ for saved files",
    "6️⃣  Add HF token for image gen API",
]
for i, step in enumerate(steps):
    y = 0.82 - i * 0.14
    ax6.text(0.05, y, step, fontsize=9, color='#e6edf3', 
             transform=ax6.transAxes, va='center')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('../outputs/dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Dashboard saved to ../outputs/dashboard.png")

In [ ]:
# ============================================================
#  CELL 18 — Final Summary
# ============================================================
print("""
╔══════════════════════════════════════════════════════════════╗
║       🤖 ALL-IN-ONE GENERATIVE AI PLAYGROUND — COMPLETE      ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ✅ Text Generation    (GPT-2)                               ║
║  ✅ Question Answering (DistilBERT)                          ║
║  ✅ Text Summarization (BART)                                ║
║  ✅ Sentiment Analysis (DistilBERT SST-2)                    ║
║  ✅ Story Generator    (GPT-2 + Genre Prompts)               ║
║  ✅ Poem Generator     (GPT-2 + Theme Prompts)               ║
║  ✅ Image Generation   (Stable Diffusion 2.1)                ║
║  ✅ Image Captioning   (BLIP)                                ║
║  ✅ Translation        (Helsinki-NLP, 7 language pairs)      ║
║  ✅ Interactive Chatbot (DialoGPT-medium)                    ║
║  ✅ Visualization Dashboard                                  ║
║                                                              ║
║  📁 All outputs saved to: ../outputs/                        ║
║  ⭐ Star this repo if you found it useful!                   ║
╚══════════════════════════════════════════════════════════════╝
""")